In [2]:
import keras
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

# Label 0 = Subject performs the action USING the object (Instrument)
# Label 1 = Target/Receiver POSSESSES or HAS the object (Attribute)

sentences = [
    # --- "astronomer / telescope" ---
    "the man used a telescope to see the astronomer",
    "looking through his telescope the man saw the astronomer",
    "the man saw the astronomer using his own telescope",
    "the man saw the astronomer who held a telescope",
    "the man saw the astronomer carrying a large telescope",
    "the astronomer with a telescope was seen by the man",
    # --- "dog / binoculars" ---
    "the girl spotted the dog through her binoculars",
    "using binoculars the girl looked at the stray dog",
    "with binoculars in hand the girl saw the dog",
    "the girl saw the dog wearing tiny binoculars",
    "the dog with binoculars strapped on was spotted",
    "the girl saw the dog that had binoculars",
    # --- "thief / umbrella" ---
    "the officer hit the thief using an umbrella",
    "swinging his umbrella the officer struck the thief",
    "the officer used an umbrella to hit the thief",
    "the officer hit the thief who carried an umbrella",
    "the thief with an umbrella was hit by the officer",
    "the officer hit the thief holding a black umbrella",
    # --- "boy / stick" ---
    "the hiker poked the bear with a long stick",
    "using a wooden stick the hiker poked the bear",
    "the hiker took a stick and poked the bear",
    "the hiker poked the bear holding a big stick",
    "the bear with a stick in paw was poked",
    "the hiker poked the bear that carried a stick",
    # --- "spy / camera" ---
    "the agent filmed the spy using a hidden camera",
    "with a small camera the agent filmed the spy",
    "the agent used a camera to film the spy",
    "the agent filmed the spy who wore a camera",
    "the spy with a camera was filmed by the agent",
    "the agent filmed the spy carrying a silver camera",
]

labels = np.array([0, 0, 0, 1, 1, 1] * 5)

In [4]:
MAX_LEN = 12
VOCAB_SIZE = 150
EMBED_DIM = 48  # Divisible by 3 heads (3 * 16 = 48)
NUM_HEADS = 3

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE, output_mode="int", output_sequence_length=MAX_LEN
)
vectorizer.adapt(sentences)
vocab = vectorizer.get_vocabulary()


class TokenAndPositionEmbedding(layers.Layer):

  def __init__(self, maxlen, vocab_size, embed_dim):
    super().__init__()
    self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
    self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

  def call(self, x):
    maxlen = tf.shape(x)[-1]
    positions = tf.range(start=0, limit=maxlen, delta=1)
    return self.token_emb(x) + self.pos_emb(positions)


# Model
inputs = layers.Input(shape=(MAX_LEN,), dtype=tf.int32)
x_embed = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)

# Multi-Head Attention with 3 heads
mha_layer = layers.MultiHeadAttention(
    num_heads=NUM_HEADS, key_dim=16, name="multi_head_attention"
)
attn_output, attn_scores = mha_layer(
    query=x_embed, value=x_embed, return_attention_scores=True
)

# Residual Connection + LayerNorm
x_contextual = layers.LayerNormalization(epsilon=1e-6)(x_embed + attn_output)

# Classification Head
pooled = layers.GlobalAveragePooling1D()(x_contextual)
outputs = layers.Dense(1, activation="sigmoid")(pooled)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Train
X_tokens = vectorizer(sentences)
model.fit(X_tokens, labels, epochs=80, verbose=0)
print("Final Accuracy:", model.evaluate(X_tokens, labels, verbose=0)[1])

Final Accuracy: 1.0
